# Satistical Language Modeling

Today we move fromm syntactic sequence modleiong into probabilistic text prediction.

Before transformer architectures, statistical language models predicted the next token by estimating:
- Historical context frequency

- Conditional word probabilities

- Maximum likelihood estimates

We will build a word-level N-Gram Language Model from scratch.

The model will:
- Extract sliding historical contexts

- Estimate conditional probabilities

- Apply Add-Alpha smoothing

- Generate text autoregressively using greedy decoding

### Objective

We will:
- Build a flexible N-Gram context extraction engine

- Train conditional probability distibutions

- Implement Add-Alpha smoothing

- Generate sequences using greedy next-token prediction

### Constraints
- No NLP frameworks

- No nltk / gensim / sklearn

- Native Python collections only

- Pure NumPy for numerical operations

- Support arbitrary N values

- Use log-space calculations for sequence probabilities

### Statistical Language Model Framework
An N-Gram model predicts the next token based on previous tokens.

For a N-Gram model of size $N$:
$$
Context = (w_{t-(N-1)}, ..., w_{t-1})
$$

The model estimates:
$$
P(w_t | Context)
$$

Using observed frequency counts.

For this notebook:
- $N = 3$ (Trigram model)

- Context size = 2 previous words

### Expected Output Layout
```
STATISTICAL LANGUAGE MODELER (v1)

TRAINING STATISTICS
Token Count: 21
Vocabulary Size: 11
Total Trigram Contexts: 19

PROBABILITY DISTRIBUTION (History: ('dense', 'matrices'))
-> capture: 0.6452
-> represent: 0.3548

GENERATION TEST
Seed Prompt: dense matrices
Generated Sequence: dense matrices capture semantic meaning via spatial proximity
```

### Imports

In [1]:
from collections import defaultdict, Counter
import numpy as np

### Corpus (Training Data)

In [ ]:
corpus_modeling = (
    "dense matrices capture semantic meaning via spatial proximity "
    "dense matrices capture statistical distributions via structural maps "
    "dense matrices represent text as vector coordinates"
)

tokens = corpus_modeling.split()

ngram_size = 3
alpha = 0.1

### Vocabulary Construction

In [ ]:
def build_vocabulary(tokens):
    "Creates vocabulary lookup tables."
    
    vocab = sorted(set(tokens))

    word_to_index = {word: i for i, word in enumerate(vocab)}
    index_to_word = {i: word for i, word in enumerate(vocab)}

    return vocab, word_to_index, index_to_word

### N-Gram Context Extraction Engine

In [5]:
def extract_ngram_contexts(tokens, n):
    "Extract (history tuple, next word) pairs for arbitraty n-gram size."

    contexts = []
    history_size = n - 1

    for i in range(len(tokens) - history_size):
        history = tuple(tokens[i:i + history_size])
        target = tokens[i + history_size]
        contexts.append((history, target))

    return contexts

### Frequency Table Builder

In [6]:
def build_ngram_counts(context_pairs):
    "Builds conditional frequency tables."

    context_counts = defaultdict(Counter)

    for context, target in context_pairs:
        context_counts[context][target] += 1

    return context_counts

### Probability Distribution Engine

In [7]:
def compute_probability_table(context_counts, vocab, alpha = 0.1):
    "Applies Add-Alpha smoothing. P(word | context)"

    probability_table = {}
    vocab_size = len(vocab)

    for context, targets in context_counts.items():
        total = sum(targets.values())
        distribution = {}
        denominator = total + (alpha * vocab_size)

        for word in vocab:
            count = targets[word]
            probability = (count + alpha) / denominator
            distribution[word] = probability

        probability_table[context] = distribution

    return probability_table

### Log Probability Engine

In [8]:
def compute_log_probability(sequence, probability_table):
    "Calulates sequence probability in log space."

    log_prob = 0.0

    for context, word in sequence:
        prob = probability_table[context][word]
        log_prob += np.log(prob)

    return log_prob

### Greedy Generation Engine

In [9]:
def generate_text(seed, probability_table, max_tokens):
    "Generates text using greedy decoding."

    generated = seed.split()
    history_size = len(generated)

    for _ in range(max_tokens):
        context = tuple(generated[-history_size:])

        if context not in probability_table:
            break

        next_word = max(
            probability_table[context], 
            key = probability_table[context].get
        )

        generated.append(next_word)

    return generated

### Analysis Helper

In [11]:
def analyze_distribution(probability_table, context, threshold = 0):
    "Extracts non-zero probability candidates for a context."

    distribution = probability_table[context]
    results = []

    for word, probability in distribution.items():
        if probability > threshold:
            results.append(
                (word, probability)
            )

    return sorted(results, key = lambda x: x[1], reverse = True)

### Evaluation Harness

In [12]:
def evaluate_ngram_model(corpus,ngram_size = 3, alpha = 0.1):
    print("STATISTICAL LANGUAGE MODELER (v1)\n")

    tokens = corpus.split()
    vocabulary, word_to_index, index_to_word = (build_vocabulary(tokens))
    contexts = extract_ngram_contexts(tokens, ngram_size)
    counts = build_ngram_counts(contexts)
    probability_table = (
        compute_probability_table(
            counts,
            vocabulary,
            alpha
        )
    )


    print("TRAINING STATISTICS")
    
    print(f"Token Count: {len(tokens)}")
    print(f"Vocabulary Size: {len(vocabulary)}")
    print(f"Total Trigram Contexts: {len(contexts)}\n")
    
    print(
        "PROBABILITY DISTRIBUTION "
        "(History: ('dense', 'matrices'))"
    )

    history = ("dense", "matrices")
    distribution = analyze_distribution(probability_table, history)

    for word, probability in distribution:
        if probability > alpha:
            print(f"-> {word}: {probability:.4f}")


    print("\nGENERATION TEST\n")
    
    seed = "dense matrices"
    generated = generate_text(seed, probability_table, max_tokens = 5)

    print(f"Seed Prompt: {seed}")
    print("Generated Sequence:")
    print(" ".join(generated))

### Execute Pipeline

In [13]:
evaluate_ngram_model(corpus_modeling, ngram_size = 3, alpha = 0.1)

STATISTICAL LANGUAGE MODELER (v1)

TRAINING STATISTICS
Token Count: 23
Vocabulary Size: 17
Total Trigram Contexts: 21

PROBABILITY DISTRIBUTION (History: ('dense', 'matrices'))
-> capture: 0.4468
-> represent: 0.2340

GENERATION TEST

Seed Prompt: dense matrices
Generated Sequence:
dense matrices capture semantic meaning via spatial
